[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sadriica/Curso_ANH/blob/main/modulo1_sig/modulo1_sig.ipynb)

Primera vez en Colab: ver la [guía](https://github.com/Sadriica/Curso_ANH/blob/main/guia_colab.md). Términos: [glosario](https://github.com/Sadriica/Curso_ANH/blob/main/glosario.md).

# Data & GIS para Energía
## Módulo 1: SIG (Sistemas de Información Geográfica)

Módulo introductorio. Antes de programar con datos, conviene entender qué es un SIG, para qué
sirve en energía, y dos temas técnicos que aparecen todo el tiempo:
la geodesia (cómo se ubican las cosas en la Tierra) y las estructuras de datos geográficos.

Contenido:

1. ¿Qué es un SIG?
2. Casos de aplicación.
3. Geodesia: coordenadas y sistemas de referencia.
4. Fuentes y estructuras de datos.

Nivel básico, corre completo en Colab. Los archivos usados están también en `recursos/`.

## 0. Preparación

In [ ]:
!pip install -q geopandas folium rasterio

In [ ]:
import pandas as pd
import geopandas as gpd
import folium
import matplotlib.pyplot as plt

## 1. ¿Qué es un SIG?

Un SIG (Sistema de Información Geográfica) es un conjunto de herramientas para guardar,
analizar y visualizar datos que tienen ubicación. Cada dato no es solo un número: es un número
en un lugar.

Un SIG organiza la información en capas (vías, ríos, viento) y cada elemento tiene dos partes:

- **Geometría**: dónde está (un punto, una línea, un polígono).
- **Atributos**: qué es o cuánto vale (nombre, población, velocidad de viento).

Ejemplo: un punto con atributos sobre un mapa.

In [ ]:
punto = gpd.GeoDataFrame(
    {"ciudad": ["Riohacha"], "viento_ms": [7.3]},
    geometry=gpd.points_from_xy([-72.91], [11.55]), crs="EPSG:4326",
)
m = folium.Map(location=[11.55, -72.91], zoom_start=8, tiles="CartoDB positron")
folium.Marker([11.55, -72.91], tooltip="Riohacha - viento 7.3 m/s").add_to(m)
m

## 2. Casos de aplicación

En el sector energético un SIG sirve, entre otros, para:

- **Ubicación de proyectos**: dónde conviene un parque eólico o solar según el recurso, el acceso
  y las restricciones.
- **Redes e infraestructura**: distancia a líneas de transmisión, subestaciones, puertos, vías.
- **Riesgo y ambiente**: zonas de amenaza (inundación, deslizamiento), áreas protegidas,
  comunidades.
- **Análisis social y territorial**: población, cobertura, ordenamiento.

El curso se centra en el primero: decidir dónde conviene un proyecto combinando varias capas.
Los módulos siguientes construyen ese análisis paso a paso.

## 3. Geodesia: coordenadas y sistemas de referencia

La Tierra no es plana ni una esfera perfecta. Para ubicar un punto se usan **coordenadas**, y
para eso hace falta un sistema de referencia (CRS). Los dos más frecuentes:

- **EPSG:4326**: latitud y longitud en **grados**. Es el estándar de GPS y de la mayoría de datos.
- **EPSG:9377**: coordenadas planas en **metros** (el oficial de Colombia). Útil para medir
  distancias y áreas.

En grados no se puede medir distancia directamente: un grado no mide lo mismo en todas partes.
Para medir hay que pasar a un sistema en metros.

In [ ]:
ciudades = gpd.GeoDataFrame(
    {"ciudad": ["Riohacha", "Valledupar"]},
    geometry=gpd.points_from_xy([-72.91, -73.25], [11.55, 10.46]), crs="EPSG:4326",
)

# distancia en grados (no es una medida real)
d_grados = ciudades.geometry.iloc[0].distance(ciudades.geometry.iloc[1])
# distancia real: reproyectar a metros y medir
cm = ciudades.to_crs("EPSG:9377")
d_metros = cm.geometry.iloc[0].distance(cm.geometry.iloc[1])

print(f"'distancia' en grados : {d_grados:.3f}  (no es una distancia real)")
print(f"distancia en km       : {d_metros/1000:.1f} km  (reproyectando a metros)")

Cuando los datos vienen en sistemas distintos, el primer paso es llevarlos todos al mismo CRS.
El Módulo 2 lo trata en detalle.

## 4. Fuentes y estructuras de datos

Los datos geográficos vienen en dos grandes estructuras:

- **Vector**: geometrías discretas con atributos. Tres tipos: punto (un pozo, una ciudad),
  línea (una vía, un río) y polígono (un municipio, un área protegida). Formatos típicos:
  Shapefile, GeoJSON, GeoPackage.
- **Raster**: una grilla de celdas, cada una con un valor. Sirve para variables continuas
  (elevación, viento, radiación). Formato típico: GeoTIFF.

Fuentes habituales en Colombia: DANE (división político-administrativa, censo), IDEAM
(clima), geoportales de entidades (energía, ambiente, riesgo).

Las dos estructuras, una al lado de la otra: un vector (puntos) y un raster pequeño.

In [ ]:
# Vector: puntos
print("VECTOR (puntos):")
print(ciudades)

In [ ]:
# Raster: grilla de elevacion sintetica
import numpy as np, rasterio
from rasterio.transform import from_bounds
W, H = 40, 40
elev = np.linspace(0, 3000, W * H).reshape(H, W).astype("float32")   # 0 a 3000 m
tr = from_bounds(-75.5, 8.0, -71.0, 12.6, W, H)
with rasterio.open("elevacion.tif", "w", driver="GTiff", height=H, width=W, count=1,
                   dtype="float32", crs="EPSG:4326", transform=tr) as dst:
    dst.write(elev, 1)

import rasterio.plot
with rasterio.open("elevacion.tif") as src:
    print("RASTER:", src.width, "x", src.height, "celdas, CRS", src.crs)
    rasterio.plot.show(src, cmap="terrain", title="Elevacion (raster de ejemplo)")

## Actividad individual

Calcule la distancia real, en kilómetros, entre Santa Marta (lat 11.24, lon -74.20) y
Barranquilla (lat 10.96, lon -74.80).

Procedimiento: arme un GeoDataFrame en EPSG:4326 con los dos puntos, reproyéctelo a EPSG:9377
(metros) y mida. Escriba su código en la celda siguiente. Más abajo está la solución.

In [ ]:
# Escriba su codigo aqui


<details><summary>Ver solución</summary>

```python
par = gpd.GeoDataFrame(
    {"ciudad": ["Santa Marta", "Barranquilla"]},
    geometry=gpd.points_from_xy([-74.20, -74.80], [11.24, 10.96]), crs="EPSG:4326",
).to_crs("EPSG:9377")
d = par.geometry.iloc[0].distance(par.geometry.iloc[1])
print(f"{d/1000:.1f} km")   # ~ 72.5 km
```
</details>

## Cierre

Resumen del módulo:

- Un SIG maneja datos con ubicación, organizados en capas de geometría y atributos.
- La geodesia define cómo se ubican las cosas; el sistema de coordenadas hay que cuidarlo.
- Los datos son vector (puntos, líneas, polígonos) o raster (grillas).

El Módulo 2 maneja estos datos con Python: leerlos, unificarlos y llevarlos a una malla común.